In [ ]:
# Fix Windows asyncio/zmq warning and ensure project root is on sys.path
import asyncio
from asyncio import WindowsSelectorEventLoopPolicy
asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())
import sys
from pathlib import Path
# ensure project root is on sys.path so `import src` works
# nbconvert executes the notebook with working dir set to the notebook's folder (data/),
# so detect that and pick the parent as the project root when appropriate.
cwd = Path.cwd()
if cwd.name == 'data' and cwd.parent.exists():
    project_root = cwd.parent
else:
    project_root = cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from src.models import train_models, evaluate_models
import joblib


In [ ]:
# Load processed features from the project data directory (handles nbconvert cwd)
from pathlib import Path
data_path = Path(project_root) / 'data' / 'processed_features.csv'
print('Reading data from', data_path)
# Fail early with a helpful message if the processed CSV is missing
if not data_path.exists():
    raise FileNotFoundError(f"{data_path} not found. Generate 'processed_features.csv' by running the feature engineering pipeline or the notebook 'data/feature_engineering.ipynb'.")
df = pd.read_csv(data_path, index_col=[0,1], parse_dates=[1])
print(df.shape)
df.head()


In [ ]:
X = df.drop(columns=["target_5d"])
y = df["target_5d"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
models = train_models(X_train_scaled, y_train)


In [ ]:
evaluate_models(models, X_test_scaled, y_test)


In [ ]:
joblib.dump(models["xgboost"], "data/best_model_xgb.pkl")
print("✅  Model saved to data/best_model_xgb.pkl")
